# Transaction Graph Analysis (GraphFrames)

**Question:** do fraud-flagged transactions form *networks* of accounts (rings, money mules, chains of transfers), or are they isolated events?

* **Vertices** = bank accounts: `gold_accounts` (column `id`)
* **Edges** = transactions: `gold_transfers` (`src` = sender → `dst` = receiver, plus amount, type, `fraud_flag`, ...)

Flow: first explore the single transactions (EDA), then the network between accounts (graph).
All tables come from `GL_01_Data_Engineering`, so run that notebook first.

In [ ]:
# GraphFrames is a separate package, not part of core Spark
%pip install graphframes-py

In [ ]:
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from graphframes import GraphFrame

## 1. EDA: do single transactions explain fraud?

Uses `silver_transactions` (all cleaned columns, including the network metrics that the graph tables leave out).

* Summary statistics of the numeric columns, and their correlation with the fraud flag
* Fraud rate per category: if fraud were driven by e.g. the device or transaction type, the bars would differ clearly from the overall rate

In [ ]:
transactions = spark.table("silver_transactions").withColumn("fraud", F.col("fraud_flag").cast("int"))
NUMERIC_COLS = ["transaction_amount", "latency_ms", "slice_bandwidth_mbps"]

transactions.select(NUMERIC_COLS).summary().show()

for col in NUMERIC_COLS:
    print(f"corr({col}, fraud) = {transactions.stat.corr(col, 'fraud'):.3f}")

In [ ]:
CATEGORY_COLS = ["transaction_type", "transaction_status", "device_used", "network_slice_id"]
overall_pct = transactions.select(F.avg("fraud")).first()[0] * 100

# Aggregate in Spark, collect only the few summary rows for plotting
fig, axes = plt.subplots(1, len(CATEGORY_COLS), figsize=(16, 3.5), sharey=True)
for ax, col in zip(axes, CATEGORY_COLS):
    rows = transactions.groupBy(col).agg(F.avg("fraud").alias("fraud_rate")).orderBy(col).collect()
    bars = ax.bar([r[col] for r in rows], [r["fraud_rate"] * 100 for r in rows], color="#d95f02", alpha=0.7)
    ax.bar_label(bars, fmt="%.1f%%")
    ax.axhline(overall_pct, color="black", linestyle="--", linewidth=1, label=f"overall {overall_pct:.1f}%")
    ax.set_title(col)
axes[0].set_ylabel("Fraud rate (%)")
axes[0].set_ylim(0, 100)
axes[0].legend(loc="upper left")
plt.tight_layout()
plt.show()

**Result:** every category sits close to the overall fraud rate (~48%), and amount, latency and bandwidth are practically uncorrelated with fraud.
The details of a single transaction do not explain fraud, so the next step is to look at the **relationships between accounts**.

## 2. Build the graph

In [ ]:
g = GraphFrame(spark.table("gold_accounts"), spark.table("gold_transfers"))

print("Accounts (vertices):", g.vertices.count())
print("Transactions (edges):", g.edges.count())
print("Overall fraud rate:", g.edges.select(F.round(F.avg(F.col("fraud_flag").cast("int")) * 100, 1)).first()[0], "%")

## 3. Degrees: how connected is each account?

`degree` = number of transactions an account takes part in (sent + received).
A money mule would show up as an account with a very high degree. We also check whether senders with more transactions are flagged more often.

In [ ]:
g.degrees.groupBy("degree").count().orderBy("degree").show()

# Fraud rate of each transaction, grouped by how active its sender is
(g.edges
    .join(g.degrees.withColumnRenamed("id", "src"), "src")
    .groupBy(F.col("degree").alias("sender_degree"))
    .agg(F.count("*").alias("transactions"),
         F.round(F.avg(F.col("fraud_flag").cast("int")) * 100, 1).alias("fraud_pct"))
    .orderBy("sender_degree")
    .show())

## 4. Connected components: are there groups of linked accounts?

Two accounts are in the same component if a path of transactions connects them (direction ignored).
A fraud ring would appear as one large component.

In [ ]:
# checkpointInterval=0: serverless compute has no checkpoint directory
components = g.connectedComponents(checkpointInterval=0)

component_sizes = components.groupBy("component").agg(F.count("*").alias("accounts"))
component_sizes.groupBy("accounts").agg(F.count("*").alias("components")).orderBy("accounts").show()

## 5. PageRank: which accounts are most central?

PageRank scores an account higher when it receives money from many accounts that are themselves central.
In a laundering network, the collecting accounts would rank on top.

In [ ]:
ranks = g.pageRank(resetProbability=0.15, maxIter=10).vertices
ranks.orderBy(F.desc("pagerank")).show(10, truncate=False)

## 6. Motifs: typical fraud patterns

* **Chain** `A → B → C`: money passed on through an intermediate account (layering)
* **Round trip** `A → B → A`: money sent back to where it came from

In [ ]:
chains = g.find("(a)-[e1]->(b); (b)-[e2]->(c)")
print("Chains A -> B -> C:", chains.count())
print("...where both transfers are fraud-flagged:", chains.where("e1.fraud_flag AND e2.fraud_flag").count())
chains.selectExpr("a.id AS sender", "b.id AS middle", "c.id AS receiver",
                  "e1.transaction_amount AS amount_1", "e2.transaction_amount AS amount_2",
                  "e1.fraud_flag AS fraud_1", "e2.fraud_flag AS fraud_2").show(truncate=False)

round_trips = g.find("(a)-[e1]->(b); (b)-[e2]->(a)")
print("Round trips A -> B -> A:", round_trips.count())

## Conclusions

* **Single transactions:** fraud rate is ~48% in every transaction type, status, device and network slice; amount, latency and bandwidth do not correlate with it.
* The network is almost completely **disconnected**: about 1,970 accounts for 1,000 transactions; almost every account appears in exactly one transaction (max degree 3).
* Connected components are tiny: ~945 isolated pairs, the largest group has only 4 accounts. **No fraud rings.**
* PageRank cannot single out hubs, since no account collects money from many others.
* Only 18 chains `A → B → C` exist (7 with both transfers flagged) and no round trips.
* ~48% of transactions are flagged, and the rate does not change with how active an account is.

**For the client:** in this data, fraud is a property of individual transactions, not of account networks. The graph pipeline (GraphFrames on Spark) is still the right tool: the same code scales to millions of accounts, where rings and mules *would* show up in the degree, component and motif analyses above.